# Model evaluation — val 2024 (Phase 2)

Deep evaluation of the LightGBM baseline from `models/train_baseline.py`:

1. Overall metrics per tier × horizon: PR-AUC, best-F2 point, vs **persistence** and the **hybrid rule** (alert if model *or* persistence fires — fixes ge30 at short horizons)
2. **Probability calibration** — undoes the negative-downsampling inflation (prior correction + isotonic), so the dashboard can show real risk %
3. **Per-station FP/FN breakdown** — which stations cause the errors (feeds the Phase-2 spatial error map)

Prerequisites: `data/training/` and `models/artifacts/` exist. Never point this at the test split — test is `final_test.ipynb`, run once at the very end.

In [2]:
! pip install lightgbm
import json
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score

pd.set_option("display.width", 160, "display.max_columns", 30)

## Configuration

In [3]:
TRAINING_DIR = Path("../data/training")
ARTIFACTS = Path("../models/artifacts")

SPLIT = "val"            # do NOT change to "test" — that's final_test.ipynb
HORIZONS = [1, 3, 6]
TIERS = [5, 15, 30]
VALID_MIN = 0.8          # min observed share of the label window
NEG_FRAC = 0.05          # must match train_baseline.py
SPW_CAP = 30.0           # must match train_baseline.py

meta = json.loads((TRAINING_DIR / "features.json").read_text())
FEATURES = meta["features"] + ["station_code"]

## Load the split

In [4]:
def load_split(name):
    label_cols = ([f"y_maxdepth_{h}h" for h in HORIZONS]
                  + [f"y_valid_{h}h" for h in HORIZONS]
                  + [f"y_ge{t}_{h}h" for h in HORIZONS for t in TIERS])
    cols = ["station_code", "site_timestamp"] + meta["features"] + label_cols
    table = pq.read_table(TRAINING_DIR / f"{name}.parquet", columns=cols)
    schema = pa.schema([pa.field(f.name, pa.float32())
                        if f.type == pa.float64() else f for f in table.schema])
    df = table.cast(schema).to_pandas(self_destruct=True)
    df["station_code"] = df["station_code"].astype("category")
    return df

val = load_split(SPLIT)
print(f"{SPLIT}: {len(val):,} rows, {val.station_code.nunique()} stations")

val: 3,451,352 rows, 100 stations


## Score every classifier

In [5]:
scores = {}          # (tier, h) -> raw model probability on rows passing VALID_MIN
masks = {}           # (tier, h) -> row mask used
for h in HORIZONS:
    m = (val[f"y_valid_{h}h"] >= VALID_MIN).to_numpy()
    for t in TIERS:
        booster = lgb.Booster(model_file=str(ARTIFACTS / f"clf_ge{t}_{h}h.txt"))
        scores[(t, h)] = booster.predict(val.loc[m, FEATURES])
        masks[(t, h)] = m
        print(f"scored ge{t}_{h}h: {m.sum():,} rows")

scored ge5_1h: 3,450,977 rows
scored ge15_1h: 3,450,977 rows
scored ge30_1h: 3,450,977 rows
scored ge5_3h: 3,450,344 rows
scored ge15_3h: 3,450,344 rows
scored ge30_3h: 3,450,344 rows
scored ge5_6h: 3,449,294 rows
scored ge15_6h: 3,449,294 rows
scored ge30_6h: 3,449,294 rows


## Metric helpers

`best_f2` sweeps thresholds. `prior_correct` analytically undoes downsampling (×NEG_FRAC) and scale_pos_weight (÷SPW_CAP) odds inflation — a first-order calibration; isotonic below is the empirical one.

In [6]:
def f2_at(y_true, pred):
    tp = int((pred & (y_true == 1)).sum())
    if tp == 0:
        return 0.0, 0.0, 0.0
    prec, rec = tp / pred.sum(), tp / (y_true == 1).sum()
    return 5 * prec * rec / (4 * prec + rec), prec, rec

def best_f2(y_true, y_prob):
    best = {"f2": 0.0, "threshold": 1.0, "precision": 0.0, "recall": 0.0}
    for thr in np.unique(np.quantile(y_prob, np.linspace(0.5, 0.9999, 300))):
        f2, prec, rec = f2_at(y_true, y_prob >= thr)
        if f2 > best["f2"]:
            best = {"f2": f2, "threshold": float(thr),
                    "precision": prec, "recall": rec}
    return best

def prior_correct(p):
    odds = p / np.clip(1 - p, 1e-9, None) * NEG_FRAC / SPW_CAP
    return odds / (1 + odds)

## 1. Overall metrics: model vs persistence vs hybrid

Hybrid rule: alert when the model score crosses its threshold **or** the station is already at/above the tier right now. Persistence knowledge is free — the model should never do worse than it.

In [7]:
rows = []
thresholds = {}      # saved for final_test.ipynb — chosen on val only
for h in HORIZONS:
    for t in TIERS:
        m = masks[(t, h)]
        y = val.loc[m, f"y_ge{t}_{h}h"].to_numpy()
        prob = scores[(t, h)]
        pers = (np.nan_to_num(val.loc[m, "fl_depth_now"].to_numpy()) >= t)

        model_pt = best_f2(y, prob)
        pers_f2, pers_p, pers_r = f2_at(y, pers)
        # hybrid: model at its own threshold OR persistence
        hyb = (prob >= model_pt["threshold"]) | pers
        hyb_f2, hyb_p, hyb_r = f2_at(y, hyb)

        thresholds[f"ge{t}_{h}h"] = model_pt["threshold"]
        rows.append({
            "target": f"ge{t}_{h}h", "val_pos": int(y.sum()),
            "pr_auc": average_precision_score(y, prob),
            "model_f2": model_pt["f2"], "model_prec": model_pt["precision"],
            "model_rec": model_pt["recall"],
            "pers_f2": pers_f2, "hybrid_f2": hyb_f2,
            "hybrid_prec": hyb_p, "hybrid_rec": hyb_r,
        })

summary = pd.DataFrame(rows).round(4)
summary["winner"] = np.where(
    summary.hybrid_f2 >= summary[["model_f2", "pers_f2"]].max(axis=1),
    "hybrid", np.where(summary.model_f2 > summary.pers_f2, "model", "pers"))
summary

,target,val_pos,pr_auc,model_f2,model_prec,model_rec,pers_f2,hybrid_f2,hybrid_prec,hybrid_rec,winner
0,ge5_1h,1664,0.4671,0.3821,0.1596,0.5865,0.4679,0.3821,0.1596,0.5865,pers
1,ge15_1h,280,0.4179,0.4195,0.3555,0.4393,0.4223,0.4195,0.3555,0.4393,pers
2,ge30_1h,40,0.0002,0.0011,0.0002,0.8000,0.3672,0.0011,0.0002,0.8000,pers
3,ge5_3h,3622,0.2198,0.2395,0.1614,0.2725,0.2289,0.2395,0.1614,0.2725,hybrid
4,ge15_3h,642,0.1774,0.2162,0.3642,0.1963,0.1989,0.2162,0.3642,0.1963,hybrid
5,ge30_3h,90,0.0084,0.0829,0.0201,0.3778,0.1724,0.0900,0.0218,0.4111,pers
6,ge5_6h,6508,0.1272,0.1534,0.1613,0.1515,0.1304,0.1534,0.1613,0.1515,hybrid
7,ge15_6h,1184,0.1018,0.1220,0.3594,0.1047,0.1106,0.1220,0.3594,0.1047,hybrid
8,ge30_6h,164,0.0868,0.1199,0.0696,0.1463,0.0966,0.1199,0.0696,0.1463,hybrid


## 2. Calibration

Isotonic regression fitted on val maps raw scores to observed frequencies. The dashboard should display `calibrator.predict(raw_score)` as risk %. (Fitting on val and reporting reliability on val is mildly optimistic — acceptable for the sprint; note it in the report.)

In [8]:
calibrators = {}
rel_rows = []
for (t, h), prob in scores.items():
    y = val.loc[masks[(t, h)], f"y_ge{t}_{h}h"].to_numpy()
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(prob, y)
    calibrators[f"ge{t}_{h}h"] = iso
    cal = iso.predict(prob)
    # reliability check on the top-risk bins (the ones that trigger alerts)
    for q in (0.999, 0.9999):
        cut = np.quantile(prob, q)
        sel = prob >= cut
        rel_rows.append({"target": f"ge{t}_{h}h", "top_bin": f"{q:.2%}",
                         "n": int(sel.sum()),
                         "mean_raw": float(prob[sel].mean()),
                         "mean_prior_corrected": float(prior_correct(prob[sel]).mean()),
                         "mean_calibrated": float(cal[sel].mean()),
                         "observed_rate": float(y[sel].mean())})

joblib.dump(calibrators, ARTIFACTS / "calibrators.joblib")
pd.DataFrame(rel_rows).round(4)

,target,top_bin,n,mean_raw,mean_prior_corrected,mean_calibrated,observed_rate
0,ge5_1h,99.90%,3451,0.9500,0.2022,0.2569,0.2553
1,ge5_1h,99.99%,347,0.9996,0.7879,0.9316,0.9251
2,ge15_1h,99.90%,3451,0.6933,0.0338,0.0494,0.0493
3,ge15_1h,99.99%,346,0.9729,0.2987,0.3603,0.3555
4,ge30_1h,99.90%,142081,1.0000,1.0000,0.0002,0.0002
5,ge30_1h,99.99%,142081,1.0000,1.0000,0.0002,0.0002
6,ge5_3h,99.90%,3451,0.9463,0.1809,0.2596,0.2585
7,ge5_3h,99.99%,346,0.9994,0.7201,0.9012,0.8902
8,ge15_3h,99.90%,3451,0.7328,0.0330,0.0504,0.0493
9,ge15_3h,99.99%,346,0.9693,0.2859,0.3671,0.3642


Read: `mean_calibrated` should track `observed_rate` closely (isotonic, by construction on val); `mean_raw` shows how inflated the uncalibrated scores are.

## 3. Per-station FP/FN breakdown

At the model's val-chosen threshold (hybrid rule applied). Sorted by misses — these stations belong on the Phase-2 error map and in the supervisor conversation about undetectable stations.

In [11]:
station_rows = []
codes_all = val["station_code"].astype(str)
for (t, h), prob in scores.items():
    m = masks[(t, h)]
    y = val.loc[m, f"y_ge{t}_{h}h"].to_numpy()
    pers = (np.nan_to_num(val.loc[m, "fl_depth_now"].to_numpy()) >= t)
    pred = (prob >= thresholds[f"ge{t}_{h}h"]) | pers
    codes = codes_all[m].to_numpy()
    df = pd.DataFrame({"station": codes, "y": y, "pred": pred})
    g = df.groupby("station").apply(
        lambda s: pd.Series({
            "pos": int(s.y.sum()),
            "tp": int((s.pred & (s.y == 1)).sum()),
            "fp": int((s.pred & (s.y == 0)).sum()),
            "fn": int((~s.pred & (s.y == 1)).sum())}),
        include_groups=False).reset_index()
    g.insert(0, "target", f"ge{t}_{h}h")
    station_rows.append(g)

stations = pd.concat(station_rows, ignore_index=True)
stations["recall"] = (stations.tp / stations.pos.replace(0, np.nan)).round(3)
stations.to_csv(ARTIFACTS / "eval_station_breakdown.csv", index=False)

focus = stations[stations.target == "ge15_6h"]
print("=== ge15_6h: worst misses (FN) ===")
print(focus.nlargest(10, "fn")[["station", "pos", "tp", "fn", "fp", "recall"]]
      .to_string(index=False))
print("\n=== ge15_6h: worst false alarms (FP) ===")
print(focus.nlargest(10, "fp")[["station", "pos", "tp", "fn", "fp"]]
      .to_string(index=False))

=== ge15_6h: worst misses (FN) ===
  station  pos  tp  fn  fp  recall
FL.SMI.01  206  19 187  17   0.092
FL.BKM.01  108  15  93  10   0.139
FL.DDG.02   82  12  70   5   0.146
FL.BKM.02   53   5  48   3   0.094
FL.BNA.04   51   3  48   7   0.059
FL.WTN.03   50   2  48   3   0.040
FL.BRK.01   52   5  47   3   0.096
FL.DDG.01   51   4  47   9   0.078
FL.KTY.05   49   2  47   3   0.041
FL.CTC.03   51   5  46   2   0.098

=== ge15_6h: worst false alarms (FP) ===
  station  pos  tp  fn  fp
FL.LSI.04   25   2  23  20
FL.SMI.01  206  19 187  17
FL.PWT.02    0   0   0  16
FL.BNA.03    0   0   0  15
FL.RTW.09    0   0   0  15
FL.SLG.02    0   0   0  13
FL.LSI.03   24   1  23  12
FL.BKM.01  108  15  93  10
FL.DDG.01   51   4  47   9
FL.BNA.04   51   3  48   7


## Save the val-eval summary (thresholds feed final_test.ipynb)

In [10]:
out = {
    "split": SPLIT,
    "thresholds": thresholds,
    "summary": summary.to_dict(orient="records"),
    "config": {"VALID_MIN": VALID_MIN, "NEG_FRAC": NEG_FRAC,
               "SPW_CAP": SPW_CAP},
}
(ARTIFACTS / "eval_summary.json").write_text(json.dumps(out, indent=2))
print("saved:", ARTIFACTS / "eval_summary.json")
print("saved:", ARTIFACTS / "calibrators.joblib")
print("saved:", ARTIFACTS / "eval_station_breakdown.csv")

saved: ../models/artifacts/eval_summary.json
saved: ../models/artifacts/calibrators.joblib
saved: ../models/artifacts/eval_station_breakdown.csv


## What to look at

- `summary`: hybrid should now win or tie everywhere, including ge30 at 1h/3h
- Reliability table: calibrated probabilities usable as dashboard risk %
- Station breakdown: a few stations usually concentrate most FNs — check whether they're the ones with high NULL rates in the quality scorecard
- Thresholds in `eval_summary.json` are frozen here; `final_test.ipynb` applies them to test-2025 unchanged